## Data Loading

In [ ]:
import numpy as np
import pandas as pd
import sqlalchemy as sqla

DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "postgres"

connection_string = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@"
    f"{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Créer un moteur SQLAlchemy
engine = sqla.create_engine(connection_string)

try:
    with engine.connect() as connection:
        print("Connected to PostgreSQL successfully!")
except Exception as error:
    print(f"Error: {error}")

connection = engine.connect()

query = """
SELECT
    td.id,
    td.departure_date,
    td.train_number,
    td.relation,
    td.train_service,
    td.ptcar_number,
    td.line_number_departure,
    td.real_time_arrival,
    td.real_time_departure,
    td.planned_time_arrival,
    td.planned_time_departure,
    td.delay_arrival,
    td.delay_departure,
    ptcar.id AS ptcar_id,
    ptcar.name AS name_travel,
    td.line_number_arrival,
    dep.name AS departure_station_name,
    arr.name AS arrival_station_name,
    ptcar.longitude AS ptcar_longitude,
    ptcar.latitude AS ptcar_latitude
FROM train_data td
JOIN STATIONS ptcar ON td.ptcar_name = ptcar.id
JOIN STATIONS dep ON td.station_departure = dep.id
JOIN STATIONS arr ON td.station_arrival = arr.id
WHERE td.station_arrival = '487'
  AND td.station_departure = '159'
ORDER BY td.id;
"""

route_data = pd.read_sql(sqla.text(query), connection)
route_data = route_data.drop(columns=["train_service", "relation"])

route_station_ids = [
    159,
    151,
    149,
    150,
    156,
    157,
    154,
    245,
    773,
    132,
    310,
    365,
    422,
    300,
    628,
    614,
    591,
    540,
    118,
    178,
    235,
    281,
    480,
    108,
    642,
    625,
    623,
    557,
    558,
    384,
    199,
    559,
    645,
    123,
    195,
    73,
    261,
    560,
    187,
    438,
    172,
    342,
    341,
    80,
    503,
    629,
    267,
    313,
    608,
    339,
    452,
    564,
    493,
    317,
    704,
    757,
    70,
]

print("Road:", len(route_station_ids))

current_trip_rows = []
complete_trip_rows = []
route_stop_index = 0

for i, row in route_data.iterrows():
    ptcar_id = row["ptcar_id"]
    if ptcar_id == route_station_ids[route_stop_index]:
        current_trip_rows.append(row)
        route_stop_index += 1
        if len(current_trip_rows) == len(route_station_ids):
            complete_trip_rows.extend(current_trip_rows)
            current_trip_rows = []
            route_stop_index = 0
    else:
        current_trip_rows = []
        route_stop_index = 0

complete_trips = pd.DataFrame(complete_trip_rows)

query = """
    SELECT * FROM TYPE_DAY;
"""

type_day = pd.read_sql(sqla.text(query), connection)
complete_trips = complete_trips.drop(
    columns=[
        "departure_station_name",
        "arrival_station_name",
        "id",
        "name_travel",
        "ptcar_number",
        "ptcar_id",
    ]
)
calendar_enriched_trips = pd.merge(
    complete_trips,
    type_day,
    left_on="departure_date",
    right_on="date",
    how="left",
)
calendar_enriched_trips.loc[
    calendar_enriched_trips.index[::57], "delay_arrival"
] = 0

calendar_enriched_trips.loc[
    calendar_enriched_trips.index[::57], "real_time_arrival"
] = calendar_enriched_trips.loc[
    calendar_enriched_trips.index[::57], "real_time_departure"
]
calendar_enriched_trips.loc[
    calendar_enriched_trips.index[::57], "planned_time_arrival"
] = calendar_enriched_trips.loc[
    calendar_enriched_trips.index[::57], "planned_time_departure"
]
calendar_enriched_trips.loc[
    calendar_enriched_trips.index[::57], "line_number_arrival"
] = calendar_enriched_trips.loc[
    calendar_enriched_trips.index[::57], "line_number_departure"
]


calendar_enriched_trips.loc[
    calendar_enriched_trips.index[56::57], "line_number_departure"
] = 162
calendar_enriched_trips.loc[
    calendar_enriched_trips.index[56::57], "line_number_arrival"
] = 162

last_stop_indices = calendar_enriched_trips.index[56::57]
calendar_enriched_trips.loc[last_stop_indices, "real_time_departure"] = (
    calendar_enriched_trips.loc[last_stop_indices, "real_time_arrival"]
)

calendar_enriched_trips.loc[last_stop_indices, "planned_time_departure"] = (
    calendar_enriched_trips.loc[last_stop_indices, "planned_time_arrival"]
)

calendar_enriched_trips.loc[last_stop_indices, "delay_departure"] = (
    calendar_enriched_trips.loc[last_stop_indices, "delay_arrival"]
)

query = """
SELECT
w.date,
EXTRACT(HOUR FROM w.hour) AS hour_of_day,
AVG(w.temperature) AS avg_temperature,
AVG(w.dewpoint) AS avg_dewpoint,
AVG(w.relative_humidity) AS avg_relative_humidity,
AVG(w.precipitation) AS avg_precipitation,
AVG(w.snowfall) AS avg_snowfall,
AVG(w.wind_direction) AS avg_wind_direction,
AVG(w.wind_speed) AS avg_wind_speed,
AVG(w.pressure) AS avg_pressure
FROM WEATHER w
GROUP BY w.date, hour_of_day
ORDER BY w.date, hour_of_day;
"""

weather = pd.read_sql(sqla.text(query), connection)

print("In Loading...")

calendar_enriched_trips["time"] = pd.to_datetime(
    calendar_enriched_trips["planned_time_arrival"], format="%H:%M:%S"
)
calendar_enriched_trips["hour"] = calendar_enriched_trips["time"].dt.hour
trip_features = pd.merge(
    calendar_enriched_trips,
    weather,
    left_on=["departure_date", "hour"],
    right_on=["date", "hour_of_day"],
    how="left",
)
trip_features = trip_features.drop(
    columns=["date_x", "date_y", "hour_of_day", "time", "hour"]
)


def parse_time_to_seconds(time_value):
    if pd.isna(time_value):
        return 0
    try:
        if isinstance(time_value, (int, float)):
            return int(time_value)
        if str(time_value).isdigit():
            return int(time_value)
        parsed_time = pd.to_datetime(
            time_value, format="%H:%M:%S", errors="coerce"
        )
        if pd.isna(parsed_time):
            return 0
        return (
            parsed_time.hour * 3600
            + parsed_time.minute * 60
            + parsed_time.second
        )
    except Exception:
        return 0


trip_features["departure_date"] = pd.to_datetime(
    trip_features["departure_date"], format="%Y-%m-%d"
)

trip_features["departure_year"] = trip_features["departure_date"].dt.year
trip_features["departure_month"] = trip_features["departure_date"].dt.month
trip_features["departure_day"] = trip_features["departure_date"].dt.day
trip_features["departure_weekday"] = trip_features["departure_date"].dt.weekday

trip_features["real_time_arrival"] = trip_features["real_time_arrival"].apply(
    parse_time_to_seconds
)
trip_features["planned_time_arrival"] = trip_features[
    "planned_time_arrival"
].apply(parse_time_to_seconds)
trip_features["planned_time_departure"] = trip_features[
    "planned_time_departure"
].apply(parse_time_to_seconds)
trip_features["real_time_departure"] = trip_features[
    "real_time_departure"
].apply(parse_time_to_seconds)

trip_features = trip_features.drop(
    columns=[
        "departure_date",
        "line_number_departure",
        "line_number_arrival",
        "train_number",
    ]
)
trip_features["delay_arrival"] = trip_features["delay_arrival"].apply(
    parse_time_to_seconds
)

valid_trips = []

for i in range(0, len(trip_features), 57):
    trip = trip_features.iloc[i: i + 57]
    if trip.shape[0] < 57:
        print("Not enough data for this trajet, skipping")
        continue

    if trip.isnull().values.any():
        print(f"Skipping trajet starting at index {i}: contains NaN")
        continue

    if trip["delay_arrival"].abs().max() > 36000:
        print("Delay arrival exceeds 10000 seconds, skipping")
        continue

    valid_trips.append(trip)

valid_trip_features = pd.concat(valid_trips, ignore_index=True)
valid_trip_features["stop_index"] = valid_trip_features.index % 57

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

stop = 57
model_features = valid_trip_features.drop(columns=["stop_index"])
features = model_features.shape[1]
trip_count = valid_trip_features.shape[0] // stop
print("Travels:", trip_count)
print("Features:", features)
print("Stop:", stop)

In [ ]:
data = model_features.to_numpy(dtype=np.float32).reshape(
    trip_count, stop, features
)
print(data.shape)
indices = np.arange(data.shape[0])

In [ ]:
train_size = int(0.8 * len(indices))
test_size = len(indices) - train_size

train_data = data[:train_size]
test_data = data[train_size:]

print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)

## Dataset

In [ ]:
class TripDataset(Dataset):
    def __init__(self, trips, input_length, target, target_index=4):
        self.trips = trips
        self.input_length = input_length
        self.target = target
        self.target_index = target_index

        feature_indices = list(range(self.trips.shape[2]))
        excluded_feature_indices = [target_index]
        self.continuous_features_idx = [
            i for i in feature_indices if i not in excluded_feature_indices
        ]

    def __len__(self):
        return self.trips.shape[0]

    def __getitem__(self, idx):
        trip = self.trips[idx]
        continuous_features = trip[: self.input_length, :].astype(np.float32)

        target_values = trip[self.target, self.target_index]

        return (
            torch.tensor(continuous_features, dtype=torch.float32),
            torch.tensor([target_values], dtype=torch.float32),
        )


class RecurrentDelayModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(RecurrentDelayModel, self).__init__()

        self.input_size = input_size

        self.rnn = nn.RNN(
            input_size=self.input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x_cont):

        sequence_output, hidden_states = self.rnn(x_cont)
        last_hidden_state = hidden_states[-1]
        return self.fc(last_hidden_state)

In [ ]:
input_length = 1
target = 1

training_dataset = TripDataset(train_data, input_length, target)
test_dataset = TripDataset(test_data, input_length, target)

model = RecurrentDelayModel(
    input_size=features, hidden_size=features * 2, num_layers=2, output_size=1
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 3

train_loader = DataLoader(training_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

In [ ]:
def train(num_epochs, model):
    print("Training...", num_epochs, "epochs", "model:", model)
    train_avg_loss = []
    test_avg_loss = []
    for epoch in range(num_epochs):
        train_loss = []
        model.train()
        for continuous_features, target_values in train_loader:
            optimizer.zero_grad()
            outputs = model(continuous_features)

            loss = criterion(outputs, target_values)
            loss.backward()
            optimizer.step()
            train_loss.append(loss.item())
        train_avg_loss.append(sum(train_loss) / len(train_loss))
        print(
            f"Epoch [{epoch + 1}/{num_epochs}], "
            f"Train Loss: {train_avg_loss[-1]:.4f}"
        )
        model.eval()
        test_losses = []
        with torch.no_grad():
            for continuous_features, target_values in test_loader:
                outputs = model(continuous_features)
                loss = criterion(outputs, target_values)
                test_losses.append(loss.item())
        test_avg_loss.append(sum(test_losses) / len(test_losses))
        print(
            f"Epoch [{epoch + 1}/{num_epochs}], "
            f"Test Loss: {test_avg_loss[-1]:.4f}"
        )

    return train_avg_loss, test_avg_loss

In [ ]:
train_avg_loss, test_avg_loss = train(num_epochs, model)